# Calculadoras de Distribuições Discretas

Este notebook contém calculadoras interativas para as principais distribuições de probabilidade discretas.

## 💡 Aproximações Possíveis

| De (Original) | Para (Aproximada) | Critério |
| :--- | :--- | :--- |
| **Hipergeométrica** | **Binomial** | $M \geq 10N$ |
| **Binomial** | **Poisson** | $N \geq 20$ e ($Np \leq 7$ ou $Nq \leq 7$) |
| **Binomial** | **Normal** | $N \geq 20, Np > 7, Nq > 7$ |
| **Poisson** | **Normal** | $\lambda > 10$ |

In [ ]:
import scipy.stats as stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt


## 1. Distribuição Binomial

A distribuição binomial modela o número de sucessos em n tentativas independentes, cada uma com probabilidade p de sucesso.

**Parâmetros:**
- n: número de tentativas
- p: probabilidade de sucesso em cada tentativa
- X: número de sucessos

In [ ]:
import scipy.stats as stats
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-darkgrid')

# Criar widgets
n_widget = widgets.IntText(value=20, description='n (tentativas):', style={'description_width': 'initial'})
p_widget = widgets.FloatText(value=0.3, description='p (probabilidade):', style={'description_width': 'initial'})

mode_widget = widgets.Dropdown(
    options=['Direto (k → P)', 'Inverso (P → k)', 'Intervalo'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial'}
)

# Widgets para modo direto
k_widget = widgets.IntText(value=6, description='k (sucessos):', style={'description_width': 'initial'})
calc_type = widgets.Dropdown(
    options=[('P(X = k)', 'pmf'), ('P(X ≤ k)', 'cdf'), ('P(X ≥ k)', 'sf'), ('P(X > k)', 'gt'), ('P(X < k)', 'lt')],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

# Widgets para modo inverso
prob_widget = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})
inverse_type = widgets.Dropdown(
    options=[('CDF: P(X ≤ k) = p', 'cdf'), ('SF: P(X ≥ k) = p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

# Widgets para modo intervalo
interval_mode = widgets.Dropdown(
    options=['Direto: P(a ≤ X ≤ b)', 'Inverso: Intervalo central'],
    value='Direto: P(a ≤ X ≤ b)',
    description='Tipo intervalo:',
    style={'description_width': 'initial'}
)
a_widget = widgets.IntText(value=4, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget = widgets.IntText(value=10, description='b (limite sup.):', style={'description_width': 'initial'})
prob_interval = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

show_graph = widgets.Checkbox(value=True, description='Mostrar gráfico')

output = widgets.Output()

def calcular_binomial(change=None):
    with output:
        clear_output(wait=True)
        try:
            n = n_widget.value
            p = p_widget.value
            q = 1 - p
            
            if n < 0 or p < 0 or p > 1:
                print("❌ Valores inválidos! Certifique-se: n ≥ 0, 0 ≤ p ≤ 1")
                return
            
            dist = stats.binom(n, p)
            
            # Modo Direto
            if mode_widget.value == 'Direto (k → P)':
                k = k_widget.value
                if k < 0 or k > n:
                    print(f"❌ Valor inválido! k deve estar entre 0 e {n}")
                    return
                
                # Calcular probabilidade
                if calc_type.value == 'pmf':
                    result = dist.pmf(k)
                    print(f"P(X = {k}) = {result:.6f}")
                elif calc_type.value == 'cdf':
                    result = dist.cdf(k)
                    print(f"P(X ≤ {k}) = {result:.6f}")
                elif calc_type.value == 'sf':
                    result = dist.sf(k-1)
                    print(f"P(X ≥ {k}) = {result:.6f}")
                elif calc_type.value == 'gt':
                    result = dist.sf(k)
                    print(f"P(X > {k}) = {result:.6f}")
                elif calc_type.value == 'lt':
                    result = dist.cdf(k-1) if k > 0 else 0
                    print(f"P(X < {k}) = {result:.6f}")
            
            # Modo Inverso
            elif mode_widget.value == 'Inverso (P → k)':
                prob = prob_widget.value
                if prob < 0 or prob > 1:
                    print("❌ Probabilidade deve estar entre 0 e 1")
                    return
                
                if inverse_type.value == 'cdf':
                    k = dist.ppf(prob)
                    print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                    print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                    print(f"k = {int(k)}")
                    print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                else:  # sf
                    k = dist.isf(prob)
                    print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                    print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                    print(f"k = {int(k)}")
                    if int(k) >= 1:
                        print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
            
            # Modo Intervalo
            elif mode_widget.value == 'Intervalo':
                if interval_mode.value == 'Direto: P(a ≤ X ≤ b)':
                    a = a_widget.value
                    b = b_widget.value
                    if a < 0 or b > n or a > b:
                        print(f"❌ Valores inválidos! 0 ≤ a ≤ b ≤ {n}")
                        return
                    
                    # Calcular P(a ≤ X ≤ b) = P(X ≤ b) - P(X < a)
                    result = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                    print(f"P({a} ≤ X ≤ {b}) = {result:.6f}")
                    print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                    print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 0 else 0:.6f}")
                
                else:  # Inverso: Intervalo central
                    prob = prob_interval.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    # Encontrar intervalo central simétrico
                    alpha = (1 - prob) / 2
                    a = int(dist.ppf(alpha))
                    b = int(dist.ppf(1 - alpha))
                    
                    actual_prob = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                    
                    print(f"Intervalo central para probabilidade ≈ {prob:.4f}:")
                    print(f"[{a}, {b}]")
                    print(f"\nP({a} ≤ X ≤ {b}) = {actual_prob:.6f}")
                    print(f"Percentis: {alpha:.4f} e {1-alpha:.4f}")
            
            # Estatísticas
            print(f"\nEsperança E(X) = {dist.mean():.4f}")
            print(f"Variância Var(X) = {dist.var():.4f}")
            print(f"Desvio Padrão σ = {dist.std():.4f}")
            
            # Verificar aproximações
            print("\n" + "="*50)
            print("🔔 AVISOS DE APROXIMAÇÃO:")
            
            # Binomial → Poisson
            if n >= 20 and (n*p <= 7 or n*q <= 7):
                lam = n * p
                print(f"\n✅ Pode aproximar para POISSON com λ = {lam:.4f}")
                print(f"   Critério: N={n} ≥ 20 e Np={n*p:.2f} ≤ 7 ou Nq={n*q:.2f} ≤ 7")
            
            # Binomial → Normal
            if n >= 20 and n*p > 7 and n*q > 7:
                mu = n * p
                sigma = np.sqrt(n * p * q)
                print(f"\n✅ Pode aproximar para NORMAL com μ = {mu:.4f}, σ = {sigma:.4f}")
                print(f"   Critério: N={n} ≥ 20, Np={n*p:.2f} > 7, Nq={n*q:.2f} > 7")
            
            if n < 20 or (n*p <= 7 and n*q <= 7):
                print("\n⚠️  Nenhuma aproximação recomendada. Use a distribuição Binomial.")
            
            # Gráfico
            if show_graph.value:
                fig, ax = plt.subplots(figsize=(10, 5))
                x_vals = np.arange(0, n+1)
                pmf_vals = dist.pmf(x_vals)
                
                if mode_widget.value == 'Intervalo' and interval_mode.value == 'Direto: P(a ≤ X ≤ b)':
                    a = a_widget.value
                    b = b_widget.value
                    colors = ['red' if a <= x <= b else 'steelblue' for x in x_vals]
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                    ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                    ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                elif mode_widget.value == 'Intervalo' and interval_mode.value == 'Inverso: Intervalo central':
                    prob = prob_interval.value
                    alpha = (1 - prob) / 2
                    a = int(dist.ppf(alpha))
                    b = int(dist.ppf(1 - alpha))
                    colors = ['red' if a <= x <= b else 'steelblue' for x in x_vals]
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                    ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                    ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                else:
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color='steelblue', edgecolor='black')
                    if mode_widget.value == 'Direto (k → P)':
                        k = k_widget.value
                        ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                    elif mode_widget.value == 'Inverso (P → k)':
                        prob = prob_widget.value
                        if inverse_type.value == 'cdf':
                            k = int(dist.ppf(prob))
                        else:
                            k = int(dist.isf(prob))
                        ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                    
                ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                
                ax.set_xlabel('Número de Sucessos (k)', fontsize=12)
                ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                ax.set_title(f'Distribuição Binomial: n={n}, p={p}', fontsize=14, fontweight='bold')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
            
        except Exception as e:
            print(f"❌ Erro: {str(e)}")

def update_visibility(change=None):
    # Remover todos os observadores
    for widget in [n_widget, p_widget, k_widget, calc_type, prob_widget, inverse_type, 
                   interval_mode, a_widget, b_widget, prob_interval, show_graph, mode_widget]:
        widget.unobserve_all()
    
    # Criar nova interface baseada no modo
    if mode_widget.value == 'Direto (k → P)':
        interface = widgets.VBox([
            widgets.HTML("<h3>📊 Calculadora Binomial</h3>"),
            n_widget, p_widget, mode_widget, calc_type, k_widget, show_graph, output
        ])
    elif mode_widget.value == 'Inverso (P → k)':
        interface = widgets.VBox([
            widgets.HTML("<h3>📊 Calculadora Binomial</h3>"),
            n_widget, p_widget, mode_widget, inverse_type, prob_widget, show_graph, output
        ])
    else:  # Intervalo
        if interval_mode.value == 'Direto: P(a ≤ X ≤ b)':
            interface = widgets.VBox([
                widgets.HTML("<h3>📊 Calculadora Binomial</h3>"),
                n_widget, p_widget, mode_widget, interval_mode, a_widget, b_widget, show_graph, output
            ])
        else:
            interface = widgets.VBox([
                widgets.HTML("<h3>📊 Calculadora Binomial</h3>"),
                n_widget, p_widget, mode_widget, interval_mode, prob_interval, show_graph, output
            ])
    
    # Re-registrar observadores
    mode_widget.observe(update_visibility, 'value')
    interval_mode.observe(update_visibility, 'value')
    n_widget.observe(calcular_binomial, 'value')
    p_widget.observe(calcular_binomial, 'value')
    k_widget.observe(calcular_binomial, 'value')
    calc_type.observe(calcular_binomial, 'value')
    prob_widget.observe(calcular_binomial, 'value')
    inverse_type.observe(calcular_binomial, 'value')
    a_widget.observe(calcular_binomial, 'value')
    b_widget.observe(calcular_binomial, 'value')
    prob_interval.observe(calcular_binomial, 'value')
    show_graph.observe(calcular_binomial, 'value')
    
    # Atualizar display
    clear_output(wait=True)
    display(interface)
    calcular_binomial()

# Inicializar
update_visibility()

## 2. Distribuição de Poisson

A distribuição de Poisson modela o número de eventos que ocorrem em um intervalo fixo de tempo ou espaço.

**Parâmetros:**
- λ (lambda): taxa média de ocorrência
- X: número de eventos

In [ ]:
# Criar widgets
lambda_widget = widgets.FloatText(value=5.0, description='λ (taxa média):', style={'description_width': 'initial'})

mode_widget_poisson = widgets.Dropdown(
    options=['Direto (k → P)', 'Inverso (P → k)', 'Intervalo'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial'}
)

# Widgets para modo direto
k_poisson = widgets.IntText(value=7, description='k (eventos):', style={'description_width': 'initial'})
calc_type_poisson = widgets.Dropdown(
    options=[('P(X = k)', 'pmf'), ('P(X ≤ k)', 'cdf'), ('P(X ≥ k)', 'sf'), ('P(X > k)', 'gt'), ('P(X < k)', 'lt')],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

# Widgets para modo inverso
prob_widget_poisson = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})
inverse_type_poisson = widgets.Dropdown(
    options=[('CDF: P(X ≤ k) = p', 'cdf'), ('SF: P(X ≥ k) = p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

# Widgets para modo intervalo
interval_mode_poisson = widgets.Dropdown(
    options=['Direto: P(a ≤ X ≤ b)', 'Inverso: Intervalo central'],
    value='Direto: P(a ≤ X ≤ b)',
    description='Tipo intervalo:',
    style={'description_width': 'initial'}
)
a_widget_poisson = widgets.IntText(value=3, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget_poisson = widgets.IntText(value=8, description='b (limite sup.):', style={'description_width': 'initial'})
prob_interval_poisson = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

show_graph_poisson = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_poisson = widgets.Output()

def calcular_poisson(change=None):
    with output_poisson:
        clear_output(wait=True)
        try:
            lam = lambda_widget.value
            
            if lam <= 0:
                print("❌ Valores inválidos! λ > 0")
                return
            
            dist = stats.poisson(lam)
            
            # Modo Direto
            if mode_widget_poisson.value == 'Direto (k → P)':
                k = k_poisson.value
                if k < 0:
                    print("❌ Valor inválido! k ≥ 0")
                    return
                
                if calc_type_poisson.value == 'pmf':
                    result = dist.pmf(k)
                    print(f"P(X = {k}) = {result:.6f}")
                elif calc_type_poisson.value == 'cdf':
                    result = dist.cdf(k)
                    print(f"P(X ≤ {k}) = {result:.6f}")
                elif calc_type_poisson.value == 'sf':
                    result = dist.sf(k-1)
                    print(f"P(X ≥ {k}) = {result:.6f}")
                elif calc_type_poisson.value == 'gt':
                    result = dist.sf(k)
                    print(f"P(X > {k}) = {result:.6f}")
                elif calc_type_poisson.value == 'lt':
                    result = dist.cdf(k-1) if k > 0 else 0
                    print(f"P(X < {k}) = {result:.6f}")
            
            # Modo Inverso
            elif mode_widget_poisson.value == 'Inverso (P → k)':
                prob = prob_widget_poisson.value
                if prob < 0 or prob > 1:
                    print("❌ Probabilidade deve estar entre 0 e 1")
                    return
                
                if inverse_type_poisson.value == 'cdf':
                    k = dist.ppf(prob)
                    print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                    print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                    print(f"k = {int(k)}")
                    print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                else:  # sf
                    k = dist.isf(prob)
                    print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                    print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                    print(f"k = {int(k)}")
                    if int(k) >= 1:
                        print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
            
            # Modo Intervalo
            elif mode_widget_poisson.value == 'Intervalo':
                if interval_mode_poisson.value == 'Direto: P(a ≤ X ≤ b)':
                    a = a_widget_poisson.value
                    b = b_widget_poisson.value
                    if a < 0 or a > b:
                        print("❌ Valores inválidos! 0 ≤ a ≤ b")
                        return
                    
                    result = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                    print(f"P({a} ≤ X ≤ {b}) = {result:.6f}")
                    print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                    print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 0 else 0:.6f}")
                
                else:  # Inverso: Intervalo central
                    prob = prob_interval_poisson.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    alpha = (1 - prob) / 2
                    a = int(dist.ppf(alpha))
                    b = int(dist.ppf(1 - alpha))
                    
                    actual_prob = dist.cdf(b) - (dist.cdf(a-1) if a > 0 else 0)
                    
                    print(f"Intervalo central para probabilidade ≈ {prob:.4f}:")
                    print(f"[{a}, {b}]")
                    print(f"\nP({a} ≤ X ≤ {b}) = {actual_prob:.6f}")
                    print(f"Percentis: {alpha:.4f} e {1-alpha:.4f}")
            
            # Estatísticas
            print(f"\nEsperança E(X) = {dist.mean():.4f}")
            print(f"Variância Var(X) = {dist.var():.4f}")
            print(f"Desvio Padrão σ = {dist.std():.4f}")
            
            # Verificar aproximação
            print("\n" + "="*50)
            print("🔔 AVISOS DE APROXIMAÇÃO:")
            
            if lam > 10:
                print(f"\n✅ Pode aproximar para NORMAL com μ = {lam:.4f}, σ = {np.sqrt(lam):.4f}")
                print(f"   Critério: λ = {lam:.4f} > 10")
            else:
                print(f"\n⚠️  λ = {lam:.4f} ≤ 10. Use a distribuição de Poisson.")
            
            # Gráfico
            if show_graph_poisson.value:
                fig, ax = plt.subplots(figsize=(10, 5))
                x_vals = np.arange(0, int(lam + 5*np.sqrt(lam)))
                pmf_vals = dist.pmf(x_vals)
                
                if mode_widget_poisson.value == 'Intervalo' and interval_mode_poisson.value == 'Direto: P(a ≤ X ≤ b)':
                    a = a_widget_poisson.value
                    b = b_widget_poisson.value
                    colors = ['red' if a <= x <= b else 'coral' for x in x_vals]
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                    ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                    ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                elif mode_widget_poisson.value == 'Intervalo' and interval_mode_poisson.value == 'Inverso: Intervalo central':
                    prob = prob_interval_poisson.value
                    alpha = (1 - prob) / 2
                    a = int(dist.ppf(alpha))
                    b = int(dist.ppf(1 - alpha))
                    colors = ['red' if a <= x <= b else 'coral' for x in x_vals]
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                    ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                    ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                else:
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color='coral', edgecolor='black')
                    if mode_widget_poisson.value == 'Direto (k → P)':
                        k = k_poisson.value
                        ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                    elif mode_widget_poisson.value == 'Inverso (P → k)':
                        prob = prob_widget_poisson.value
                        if inverse_type_poisson.value == 'cdf':
                            k = int(dist.ppf(prob))
                        else:
                            k = int(dist.isf(prob))
                        ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                
                ax.axvline(lam, color='green', linestyle='--', linewidth=2, label=f'λ = {lam:.2f}')
                
                ax.set_xlabel('Número de Eventos (k)', fontsize=12)
                ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                ax.set_title(f'Distribuição de Poisson: λ={lam}', fontsize=14, fontweight='bold')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
            
        except Exception as e:
            print(f"❌ Erro: {str(e)}")

def update_visibility_poisson(change=None):
    for widget in [lambda_widget, k_poisson, calc_type_poisson, prob_widget_poisson, inverse_type_poisson,
                   interval_mode_poisson, a_widget_poisson, b_widget_poisson, prob_interval_poisson, 
                   show_graph_poisson, mode_widget_poisson]:
        widget.unobserve_all()
    
    if mode_widget_poisson.value == 'Direto (k → P)':
        interface = widgets.VBox([
            widgets.HTML("<h3>📊 Calculadora Poisson</h3>"),
            lambda_widget, mode_widget_poisson, calc_type_poisson, k_poisson, show_graph_poisson, output_poisson
        ])
    elif mode_widget_poisson.value == 'Inverso (P → k)':
        interface = widgets.VBox([
            widgets.HTML("<h3>📊 Calculadora Poisson</h3>"),
            lambda_widget, mode_widget_poisson, inverse_type_poisson, prob_widget_poisson, show_graph_poisson, output_poisson
        ])
    else:  # Intervalo
        if interval_mode_poisson.value == 'Direto: P(a ≤ X ≤ b)':
            interface = widgets.VBox([
                widgets.HTML("<h3>📊 Calculadora Poisson</h3>"),
                lambda_widget, mode_widget_poisson, interval_mode_poisson, a_widget_poisson, b_widget_poisson, 
                show_graph_poisson, output_poisson
            ])
        else:
            interface = widgets.VBox([
                widgets.HTML("<h3>📊 Calculadora Poisson</h3>"),
                lambda_widget, mode_widget_poisson, interval_mode_poisson, prob_interval_poisson, 
                show_graph_poisson, output_poisson
            ])
    
    mode_widget_poisson.observe(update_visibility_poisson, 'value')
    interval_mode_poisson.observe(update_visibility_poisson, 'value')
    lambda_widget.observe(calcular_poisson, 'value')
    k_poisson.observe(calcular_poisson, 'value')
    calc_type_poisson.observe(calcular_poisson, 'value')
    prob_widget_poisson.observe(calcular_poisson, 'value')
    inverse_type_poisson.observe(calcular_poisson, 'value')
    a_widget_poisson.observe(calcular_poisson, 'value')
    b_widget_poisson.observe(calcular_poisson, 'value')
    prob_interval_poisson.observe(calcular_poisson, 'value')
    show_graph_poisson.observe(calcular_poisson, 'value')
    
    clear_output(wait=True)
    display(interface)
    calcular_poisson()

update_visibility_poisson()

## 3. Distribuição Geométrica

A distribuição geométrica modela o número de tentativas até o primeiro sucesso.

**Parâmetros:**
- p: probabilidade de sucesso
- X: número de tentativas até o primeiro sucesso

In [ ]:
# Criar widgets
p_geom = widgets.FloatText(value=0.3, description='p (probabilidade):', style={'description_width': 'initial'})

mode_widget_geom = widgets.Dropdown(
    options=['Direto (k → P)', 'Inverso (P → k)', 'Intervalo'],
    value='Direto (k → P)',
    description='Modo:',
    style={'description_width': 'initial'}
)

# Widgets para modo direto
k_geom = widgets.IntText(value=5, description='k (tentativas):', style={'description_width': 'initial'})
calc_type_geom = widgets.Dropdown(
    options=[('P(X = k)', 'pmf'), ('P(X ≤ k)', 'cdf'), ('P(X ≥ k)', 'sf'), ('P(X > k)', 'gt')],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

# Widgets para modo inverso
prob_widget_geom = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})
inverse_type_geom = widgets.Dropdown(
    options=[('CDF: P(X ≤ k) = p', 'cdf'), ('SF: P(X ≥ k) = p', 'sf')],
    value='cdf',
    description='Tipo:',
    style={'description_width': 'initial'}
)

# Widgets para modo intervalo
interval_mode_geom = widgets.Dropdown(
    options=['Direto: P(a ≤ X ≤ b)', 'Inverso: Intervalo central'],
    value='Direto: P(a ≤ X ≤ b)',
    description='Tipo intervalo:',
    style={'description_width': 'initial'}
)
a_widget_geom = widgets.IntText(value=2, description='a (limite inf.):', style={'description_width': 'initial'})
b_widget_geom = widgets.IntText(value=8, description='b (limite sup.):', style={'description_width': 'initial'})
prob_interval_geom = widgets.FloatText(value=0.95, description='Probabilidade:', style={'description_width': 'initial'})

show_graph_geom = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_geom = widgets.Output()

def calcular_geometrica(change=None):
    with output_geom:
        clear_output(wait=True)
        try:
            p = p_geom.value
            
            if p <= 0 or p > 1:
                print("❌ Valores inválidos! 0 < p ≤ 1")
                return
            
            dist = stats.geom(p)
            
            # Modo Direto
            if mode_widget_geom.value == 'Direto (k → P)':
                k = k_geom.value
                if k < 1:
                    print("❌ Valor inválido! k ≥ 1")
                    return
                
                if calc_type_geom.value == 'pmf':
                    result = dist.pmf(k)
                    print(f"P(X = {k}) = {result:.6f}")
                elif calc_type_geom.value == 'cdf':
                    result = dist.cdf(k)
                    print(f"P(X ≤ {k}) = {result:.6f}")
                elif calc_type_geom.value == 'sf':
                    result = dist.sf(k-1)
                    print(f"P(X ≥ {k}) = {result:.6f}")
                elif calc_type_geom.value == 'gt':
                    result = dist.sf(k)
                    print(f"P(X > {k}) = {result:.6f}")
            
            # Modo Inverso
            elif mode_widget_geom.value == 'Inverso (P → k)':
                prob = prob_widget_geom.value
                if prob < 0 or prob > 1:
                    print("❌ Probabilidade deve estar entre 0 e 1")
                    return
                
                if inverse_type_geom.value == 'cdf':
                    k = dist.ppf(prob)
                    print(f"ℹ️  Para distribuições discretas, ppf retorna o menor k tal que P(X ≤ k) ≥ p")
                    print(f"\nPara P(X ≤ k) ≥ {prob:.6f}:")
                    print(f"k = {int(k)}")
                    print(f"P(X ≤ {int(k)}) = {dist.cdf(int(k)):.6f}")
                else:  # sf
                    k = dist.isf(prob)
                    print(f"ℹ️  Para distribuições discretas, isf retorna o menor k tal que P(X ≥ k) ≤ p")
                    print(f"\nPara P(X ≥ k) ≤ {prob:.6f}:")
                    print(f"k = {int(k)}")
                    if int(k) >= 1:
                        print(f"P(X ≥ {int(k)}) = {dist.sf(int(k)-1):.6f}")
            
            # Modo Intervalo
            elif mode_widget_geom.value == 'Intervalo':
                if interval_mode_geom.value == 'Direto: P(a ≤ X ≤ b)':
                    a = a_widget_geom.value
                    b = b_widget_geom.value
                    if a < 1 or a > b:
                        print("❌ Valores inválidos! 1 ≤ a ≤ b")
                        return
                    
                    result = dist.cdf(b) - (dist.cdf(a-1) if a > 1 else 0)
                    print(f"P({a} ≤ X ≤ {b}) = {result:.6f}")
                    print(f"\nCálculo: P(X ≤ {b}) - P(X < {a})")
                    print(f"       = {dist.cdf(b):.6f} - {dist.cdf(a-1) if a > 1 else 0:.6f}")
                
                else:  # Inverso: Intervalo central
                    prob = prob_interval_geom.value
                    if prob < 0 or prob > 1:
                        print("❌ Probabilidade deve estar entre 0 e 1")
                        return
                    
                    alpha = (1 - prob) / 2
                    a = int(dist.ppf(alpha))
                    b = int(dist.ppf(1 - alpha))
                    
                    actual_prob = dist.cdf(b) - (dist.cdf(a-1) if a > 1 else 0)
                    
                    print(f"Intervalo central para probabilidade ≈ {prob:.4f}:")
                    print(f"[{a}, {b}]")
                    print(f"\nP({a} ≤ X ≤ {b}) = {actual_prob:.6f}")
                    print(f"Percentis: {alpha:.4f} e {1-alpha:.4f}")
            
            # Estatísticas
            print(f"\nEsperança E(X) = {dist.mean():.4f}")
            print(f"Variância Var(X) = {dist.var():.4f}")
            print(f"Desvio Padrão σ = {dist.std():.4f}")
            
            # Gráfico
            if show_graph_geom.value:
                fig, ax = plt.subplots(figsize=(10, 5))
                x_vals = np.arange(1, min(50, int(dist.mean() + 5*dist.std())))
                pmf_vals = dist.pmf(x_vals)
                
                if mode_widget_geom.value == 'Intervalo' and interval_mode_geom.value == 'Direto: P(a ≤ X ≤ b)':
                    a = a_widget_geom.value
                    b = b_widget_geom.value
                    colors = ['red' if a <= x <= b else 'mediumseagreen' for x in x_vals]
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                    ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                    ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                elif mode_widget_geom.value == 'Intervalo' and interval_mode_geom.value == 'Inverso: Intervalo central':
                    prob = prob_interval_geom.value
                    alpha = (1 - prob) / 2
                    a = int(dist.ppf(alpha))
                    b = int(dist.ppf(1 - alpha))
                    colors = ['red' if a <= x <= b else 'mediumseagreen' for x in x_vals]
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color=colors, edgecolor='black')
                    ax.axvline(a, color='darkred', linestyle='--', linewidth=2, label=f'a = {a}')
                    ax.axvline(b, color='darkred', linestyle='--', linewidth=2, label=f'b = {b}')
                else:
                    ax.bar(x_vals, pmf_vals, alpha=0.7, color='mediumseagreen', edgecolor='black')
                    if mode_widget_geom.value == 'Direto (k → P)':
                        k = k_geom.value
                        ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                    elif mode_widget_geom.value == 'Inverso (P → k)':
                        prob = prob_widget_geom.value
                        if inverse_type_geom.value == 'cdf':
                            k = int(dist.ppf(prob))
                        else:
                            k = int(dist.isf(prob))
                        ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {int(k)}')
                
                ax.axvline(dist.mean(), color='orange', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                
                ax.set_xlabel('Número de Tentativas (k)', fontsize=12)
                ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                ax.set_title(f'Distribuição Geométrica: p={p}', fontsize=14, fontweight='bold')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
            
        except Exception as e:
            print(f"❌ Erro: {str(e)}")

def update_visibility_geom(change=None):
    for widget in [p_geom, k_geom, calc_type_geom, prob_widget_geom, inverse_type_geom,
                   interval_mode_geom, a_widget_geom, b_widget_geom, prob_interval_geom, 
                   show_graph_geom, mode_widget_geom]:
        widget.unobserve_all()
    
    if mode_widget_geom.value == 'Direto (k → P)':
        interface = widgets.VBox([
            widgets.HTML("<h3>📊 Calculadora Geométrica</h3>"),
            p_geom, mode_widget_geom, calc_type_geom, k_geom, show_graph_geom, output_geom
        ])
    elif mode_widget_geom.value == 'Inverso (P → k)':
        interface = widgets.VBox([
            widgets.HTML("<h3>📊 Calculadora Geométrica</h3>"),
            p_geom, mode_widget_geom, inverse_type_geom, prob_widget_geom, show_graph_geom, output_geom
        ])
    else:  # Intervalo
        if interval_mode_geom.value == 'Direto: P(a ≤ X ≤ b)':
            interface = widgets.VBox([
                widgets.HTML("<h3>📊 Calculadora Geométrica</h3>"),
                p_geom, mode_widget_geom, interval_mode_geom, a_widget_geom, b_widget_geom, 
                show_graph_geom, output_geom
            ])
        else:
            interface = widgets.VBox([
                widgets.HTML("<h3>📊 Calculadora Geométrica</h3>"),
                p_geom, mode_widget_geom, interval_mode_geom, prob_interval_geom, 
                show_graph_geom, output_geom
            ])
    
    mode_widget_geom.observe(update_visibility_geom, 'value')
    interval_mode_geom.observe(update_visibility_geom, 'value')
    p_geom.observe(calcular_geometrica, 'value')
    k_geom.observe(calcular_geometrica, 'value')
    calc_type_geom.observe(calcular_geometrica, 'value')
    prob_widget_geom.observe(calcular_geometrica, 'value')
    inverse_type_geom.observe(calcular_geometrica, 'value')
    a_widget_geom.observe(calcular_geometrica, 'value')
    b_widget_geom.observe(calcular_geometrica, 'value')
    prob_interval_geom.observe(calcular_geometrica, 'value')
    show_graph_geom.observe(calcular_geometrica, 'value')
    
    clear_output(wait=True)
    display(interface)
    calcular_geometrica()

update_visibility_geom()

## 4. Distribuição Hipergeométrica

A distribuição hipergeométrica modela o número de sucessos em uma amostra sem reposição.

**Parâmetros:**
- M: tamanho da população
- K: número de sucessos na população
- N: tamanho da amostra
- X: número de sucessos na amostra

In [ ]:
# Criar widgets
M_hyper = widgets.IntText(value=100, description='M (população):', style={'description_width': 'initial'})
K_hyper = widgets.IntText(value=30, description='K (sucessos pop.):', style={'description_width': 'initial'})
N_hyper = widgets.IntText(value=20, description='N (amostra):', style={'description_width': 'initial'})
k_hyper = widgets.IntText(value=8, description='k (sucessos):', style={'description_width': 'initial'})

calc_type_hyper = widgets.Dropdown(
    options=[('P(X = k)', 'pmf'), ('P(X ≤ k)', 'cdf'), ('P(X ≥ k)', 'sf')],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_hyper = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_hyper = widgets.Output()

def calcular_hipergeometrica(change=None):
    with output_hyper:
        clear_output(wait=True)
        try:
            M = M_hyper.value
            K = K_hyper.value
            N = N_hyper.value
            k = k_hyper.value
            
            if M < 0 or K < 0 or N < 0 or k < 0 or K > M or N > M:
                print("❌ Valores inválidos! K ≤ M, N ≤ M, k ≥ 0")
                return
            
            dist = stats.hypergeom(M, K, N)
            
            if calc_type_hyper.value == 'pmf':
                result = dist.pmf(k)
                print(f"P(X = {k}) = {result:.6f}")
            elif calc_type_hyper.value == 'cdf':
                result = dist.cdf(k)
                print(f"P(X ≤ {k}) = {result:.6f}")
            elif calc_type_hyper.value == 'sf':
                result = dist.sf(k-1)
                print(f"P(X ≥ {k}) = {result:.6f}")
            
            print(f"\nEsperança E(X) = {dist.mean():.4f}")
            print(f"Variância Var(X) = {dist.var():.4f}")
            print(f"Desvio Padrão σ = {dist.std():.4f}")
            
            # Verificar aproximação
            print("\n" + "="*50)
            print("🔔 AVISOS DE APROXIMAÇÃO:")
            
            # Hipergeométrica → Binomial
            if M >= 10*N:
                p_approx = K / M
                print(f"\n✅ Pode aproximar para BINOMIAL com n={N}, p={p_approx:.4f}")
                print(f"   Critério: M={M} ≥ 10N={10*N}")
                print(f"   (População grande o suficiente para considerar independência)")
            else:
                print(f"\n⚠️  M={M} < 10N={10*N}. Use a distribuição Hipergeométrica.")
            
            # Gráfico
            if show_graph_hyper.value:
                fig, ax = plt.subplots(figsize=(10, 5))
                x_vals = np.arange(max(0, N-M+K), min(K, N)+1)
                pmf_vals = dist.pmf(x_vals)
                
                ax.bar(x_vals, pmf_vals, alpha=0.7, color='mediumpurple', edgecolor='black')
                ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                
                ax.set_xlabel('Número de Sucessos na Amostra (k)', fontsize=12)
                ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                ax.set_title(f'Distribuição Hipergeométrica: M={M}, K={K}, N={N}', fontsize=14, fontweight='bold')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
            
        except Exception as e:
            print(f"❌ Erro: {str(e)}")

for widget in [M_hyper, K_hyper, N_hyper, k_hyper, calc_type_hyper, show_graph_hyper]:
    widget.unobserve_all()

M_hyper.observe(calcular_hipergeometrica, 'value')
K_hyper.observe(calcular_hipergeometrica, 'value')
N_hyper.observe(calcular_hipergeometrica, 'value')
k_hyper.observe(calcular_hipergeometrica, 'value')
calc_type_hyper.observe(calcular_hipergeometrica, 'value')
show_graph_hyper.observe(calcular_hipergeometrica, 'value')

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Hipergeométrica</h3>"),
    M_hyper, K_hyper, N_hyper, calc_type_hyper, k_hyper, show_graph_hyper, output_hyper
]))

calcular_hipergeometrica()

## 5. Distribuição Binomial Negativa

A distribuição binomial negativa modela o número de falhas antes de obter r sucessos.

**Parâmetros:**
- r: número de sucessos desejados
- p: probabilidade de sucesso
- X: número de falhas

In [ ]:
# Criar widgets
r_nbinom = widgets.IntText(value=5, description='r (sucessos):', style={'description_width': 'initial'})
p_nbinom = widgets.FloatText(value=0.4, description='p (probabilidade):', style={'description_width': 'initial'})
k_nbinom = widgets.IntText(value=7, description='k (falhas):', style={'description_width': 'initial'})

calc_type_nbinom = widgets.Dropdown(
    options=[('P(X = k)', 'pmf'), ('P(X ≤ k)', 'cdf'), ('P(X ≥ k)', 'sf')],
    value='pmf',
    description='Tipo de cálculo:',
    style={'description_width': 'initial'}
)

show_graph_nbinom = widgets.Checkbox(value=True, description='Mostrar gráfico')

output_nbinom = widgets.Output()

def calcular_nbinom(change=None):
    with output_nbinom:
        clear_output(wait=True)
        try:
            r = r_nbinom.value
            p = p_nbinom.value
            k = k_nbinom.value
            
            if r < 1 or p <= 0 or p >= 1 or k < 0:
                print("❌ Valores inválidos! r ≥ 1, 0 < p < 1, k ≥ 0")
                return
            
            dist = stats.nbinom(r, p)
            
            if calc_type_nbinom.value == 'pmf':
                result = dist.pmf(k)
                print(f"P(X = {k}) = {result:.6f}")
            elif calc_type_nbinom.value == 'cdf':
                result = dist.cdf(k)
                print(f"P(X ≤ {k}) = {result:.6f}")
            elif calc_type_nbinom.value == 'sf':
                result = dist.sf(k-1)
                print(f"P(X ≥ {k}) = {result:.6f}")
            
            print(f"\nEsperança E(X) = {dist.mean():.4f}")
            print(f"Variância Var(X) = {dist.var():.4f}")
            print(f"Desvio Padrão σ = {dist.std():.4f}")
            
            # Gráfico
            if show_graph_nbinom.value:
                fig, ax = plt.subplots(figsize=(10, 5))
                x_vals = np.arange(0, int(dist.mean() + 5*dist.std()))
                pmf_vals = dist.pmf(x_vals)
                
                ax.bar(x_vals, pmf_vals, alpha=0.7, color='goldenrod', edgecolor='black')
                ax.axvline(k, color='red', linestyle='--', linewidth=2, label=f'k = {k}')
                ax.axvline(dist.mean(), color='green', linestyle='--', linewidth=2, label=f'μ = {dist.mean():.2f}')
                
                ax.set_xlabel('Número de Falhas (k)', fontsize=12)
                ax.set_ylabel('Probabilidade P(X = k)', fontsize=12)
                ax.set_title(f'Distribuição Binomial Negativa: r={r}, p={p}', fontsize=14, fontweight='bold')
                ax.legend()
                ax.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()
            
        except Exception as e:
            print(f"❌ Erro: {str(e)}")

for widget in [r_nbinom, p_nbinom, k_nbinom, calc_type_nbinom, show_graph_nbinom]:
    widget.unobserve_all()

r_nbinom.observe(calcular_nbinom, 'value')
p_nbinom.observe(calcular_nbinom, 'value')
k_nbinom.observe(calcular_nbinom, 'value')
calc_type_nbinom.observe(calcular_nbinom, 'value')
show_graph_nbinom.observe(calcular_nbinom, 'value')

display(widgets.VBox([
    widgets.HTML("<h3>📊 Calculadora Binomial Negativa</h3>"),
    r_nbinom, p_nbinom, calc_type_nbinom, k_nbinom, show_graph_nbinom, output_nbinom
]))

calcular_nbinom()